# S2·MIÉ — RAG mínimo en ~60 líneas

**MMIA 6013 · Semana 2 · Miércoles**

Pipeline completo: chunking → índice → retrieval → prompt con contexto → LLM.

El último peldaño —generar— tiene tres respaldos y **dice cuál usó**: la API de OpenAI
si hay `OPENAI_API_KEY` en el `.env` de la raíz, un modelo local por Ollama si no, y una
respuesta pre-capturada si tampoco. En «modo inspección» el cuaderno enseña el prompt
armado, así que el retrieval —que es lo que se evalúa hoy— se depura igual sin clave.

## 🧑‍🏫 Notas docentes

- El corpus sintético ("NimbusSoft") está diseñado con trampas: una idea que
  cruza la frontera de chunks con tamaño 300 y una pregunta sin respuesta.
- Dedicar tiempo a la pregunta SIN respuesta: que vean el "no tengo información"
  como comportamiento correcto.
- Si hay sentence-transformers instalado, el retrieval es semántico real;
  con el fallback TF-IDF también funciona (corpus comparte vocabulario).
- **La generación pasó de Anthropic a OpenAI el 2026-09-23**, con `gpt-5.6-terra` fijado
  en una constante. Es una excepción consciente a la regla del curso de leer los nombres
  de `fuentes/modelos/modelos-2026-1.json`: aquí el id vive en `MODELO_API`, arriba del
  todo, y cambiarlo por `gpt-5.6-sol` o `gpt-5.6-luna` es editar una línea. Lo que **no**
  se puede cambiar sin romperlo son los dos parámetros: esta familia exige
  `max_completion_tokens` en vez de `max_tokens`, y rechaza cualquier `temperature`
  distinta de su valor por defecto — las dos cosas devuelven 400, comprobadas contra la
  API. El `.ipynb` del profesor tenía
  ese cambio hecho A MANO y a medias —comprobaba `OPENAI_API_KEY`, importaba `openai` y
  llamaba a `anthropic.Anthropic()`, que no estaba importado—, así que el peldaño de API
  nunca corrió: el `NameError` lo tragaba el `except` y caía a Ollama en silencio. Los
  notebooks se regeneran desde `_fuentes/`; editarlos a mano dura hasta el siguiente
  `build_notebooks.py`.
- Para usar la H200 en vez de un Ollama local, `_ollama(prompt, host="http://172.28.230.10:11434")`.
  Ver `curso/actividades/semana-2/h200-usfq-profesor.ipynb`.

In [1]:
import json
import os
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np


def _raiz_repo(inicio: Path) -> Path:
    """Sube directorios hasta la raíz del repositorio del curso.

    La ruta relativa fija falla: desde `curso/actividades/semana-N/` dos niveles
    arriba es `curso/`, no la raíz. Buscar hacia arriba hace además que el
    notebook corra desde donde se lo abra.
    """
    for base in [inicio, *inicio.parents]:
        if (base / "curso/actividades").is_dir():
            return base
    raise FileNotFoundError(
        "No encuentro `curso/actividades/`: abre el notebook dentro del "
        "repositorio del curso.")


RAIZ = _raiz_repo(Path.cwd())

# La clave se lee del `.env` de la raíz del repositorio, igual que en la semana 1.
# NUNCA se escribe aquí: `curso/politica-credenciales.md` lo prohíbe, y una salida
# guardada dentro del .ipynb es exactamente el sitio donde no puede aparecer.
_env = RAIZ / ".env"
if _env.is_file():
    for _linea in _env.read_text(encoding="utf-8").splitlines():
        _linea = _linea.strip()
        if _linea and not _linea.startswith("#") and "=" in _linea:
            _k, _, _v = _linea.removeprefix("export ").partition("=")
            _k, _v = _k.strip(), _v.strip().strip('"').strip("'")
            if _k and _v:
                os.environ.setdefault(_k, _v)   # lo exportado en la terminal manda

# Un solo sitio donde cambiar el modelo. La familia gpt-5.6 tiene tres hermanos
# —`terra`, `sol` y `luna`— y los tres sirven para esto; se cambia aquí y ya.
MODELO_API = "gpt-5.6-terra"
MODELO_LOCAL = "llama3.1:8b"    # el que se usa si no hay clave

# Dos parámetros que esta familia NO admite, y conviene saberlo antes de que dé 400
# (comprobado contra la API el 2026-09-23):
#
#   max_tokens        -> 400: «Unsupported parameter: use 'max_completion_tokens'»
#   temperature=0.0   -> 400: «Only the default (1) value is supported»
#
# Lo segundo tiene consecuencias para un RAG y no es un detalle de formato: **no se
# puede fijar la temperatura a cero**, así que dos corridas con el mismo contexto
# pueden redactar distinto. Lo que debe mantenerse estable es de dónde sale la
# respuesta —los chunks recuperados—, y eso es justo lo que mide esta sesión.
MAX_SALIDA = 800   # el presupuesto incluye el razonamiento, si el modelo razona

print(f"raíz del curso : {RAIZ}")
print(f"modelo de API  : {MODELO_API}")
print(f"OPENAI_API_KEY : {'presente' if os.environ.get('OPENAI_API_KEY') else 'ausente — se usa Ollama o modo inspección'}")

raíz del curso : /Users/driofrio/Agents/GenAIV3
modelo de API  : gpt-5.6-terra
OPENAI_API_KEY : presente


In [2]:
# Corpus: documentación interna de una empresa ficticia
DOCS = {
    "politica_vacaciones.md": """Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo acumulan 1.5 días de vacaciones por mes trabajado,
hasta un máximo de 18 días por año. Las vacaciones deben solicitarse con al menos
15 días de anticipación a través del portal interno. Los días no utilizados pueden
transferirse al año siguiente hasta un máximo de 5 días. Durante el primer año,
los días solo pueden tomarse después de superar el período de prueba de 3 meses.""",
    "politica_remoto.md": """Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permitido hasta 3 días por semana para todos los roles
excepto soporte de infraestructura on-site. Los días remotos se coordinan con el
líder de equipo. Para trabajar desde el exterior del país se requiere aprobación
de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.""",
    "gastos.md": """Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se reembolsan presentando factura dentro de los 30 días
posteriores al gasto. El límite diario de alimentación en viajes es de 45 USD.
Los pasajes aéreos deben comprarse en clase económica salvo vuelos de más de
8 horas, donde se permite económica premium con aprobación del gerente de área.""",
}

## Parte 1 — Chunking

### ✏️ Ejercicio 1.1
Implementa `chunk_fijo(texto, tamano=300, overlap=80)` sobre caracteres.
Devuelve una lista de strings. Luego observa si algún chunk corta una idea.

In [3]:
# Ejercicio 1.1 — chunking fijo con solapamiento
def chunk_fijo(texto: str, tamano: int = 300, overlap: int = 80):
    chunks, paso = [], tamano - overlap
    for ini in range(0, len(texto), paso):
        trozo = texto[ini:ini + tamano].strip()
        if trozo:
            chunks.append(trozo)
        if ini + tamano >= len(texto):
            break
    return chunks

In [4]:
# El corpus fragmentado. Vive FUERA del ejercicio porque las Partes 2 y 3 lo consumen:
# si estuviera dentro de la solución del 1.1, la versión de estudiante no lo tendría.
chunks, origen = [], []
for nombre, texto in DOCS.items():
    for c in chunk_fijo(texto):
        chunks.append(c)
        origen.append(nombre)

for i, c in enumerate(chunks):
    print(f"[{i}] ({origen[i]}) {c[:70]}…")

[0] (politica_vacaciones.md) Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo …
[1] (politica_vacaciones.md) pación a través del portal interno. Los días no utilizados pueden
tran…
[2] (politica_remoto.md) Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permi…
[3] (politica_remoto.md) ra trabajar desde el exterior del país se requiere aprobación
de Recur…
[4] (gastos.md) Política de reembolso de gastos de NimbusSoft.
Los gastos de viaje se …
[5] (gastos.md) deben comprarse en clase económica salvo vuelos de más de
8 horas, don…


Mira los cortes: con `overlap=80`, la idea "hasta un máximo de 5 días" transferibles
aparece completa en algún chunk aunque otro la corte. Prueba `overlap=0` y
vuelve a mirar.

## Parte 2 — Índice (embeddings + kNN)

In [5]:
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    def embed(ts):
        return np.asarray(_m.encode(ts, normalize_embeddings=True))
    MOTOR = "denso (sentence-transformers)"
    V = embed(chunks)
    def vec_consulta(q):
        return embed([q])[0]
except ImportError:
    from sklearn.feature_extraction.text import TfidfVectorizer
    _tf = TfidfVectorizer().fit(chunks)
    Vs = _tf.transform(chunks).toarray()
    V = Vs / (np.linalg.norm(Vs, axis=1, keepdims=True) + 1e-9)
    def vec_consulta(q):
        v = _tf.transform([q]).toarray()[0]
        return v / (np.linalg.norm(v) + 1e-9)
    MOTOR = "léxico (TF-IDF fallback)"

print("Retrieval:", MOTOR, "| índice:", V.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Retrieval: denso (sentence-transformers) | índice: (6, 384)


### El techo del modelo, medido

El chunking de la Parte 1 cuenta **caracteres** y el modelo de embeddings trunca en
**tokens**: el MiniLM multilingüe procesa como mucho `max_seq_length` tokens de cada
texto y descarta el resto **sin avisar**. Trescientos caracteres probablemente caben,
pero «probablemente» no es una medición. Esta celda cuenta los tokens del fragmento más
largo con el tokenizador del mismo modelo y los compara con su tope.

El Lab 02 del sábado hace esto por ti: fragmenta directamente en tokens de su modelo
—`bge-m3`, servido en la H200 de la USFQ, con un tope de 8192— y avisa si pides un
fragmento que no cabe. El MiniLM de este cuaderno es el que usa la Parte 0.b del taller
para ver el truncado con tus propios ojos.

In [6]:
if MOTOR.startswith("denso"):
    tope = int(_m.max_seq_length)
    tokens = [len(_m.tokenizer(c)["input_ids"]) for c in chunks]
    mas_largo = int(np.argmax(tokens))
    print(f"tope del modelo: {tope} tokens")
    print(f"fragmento más largo: [{mas_largo}] con {tokens[mas_largo]} tokens "
          f"({len(chunks[mas_largo])} caracteres)")
    print("cabe entero" if tokens[mas_largo] <= tope else
          f"NO cabe: se pierden {tokens[mas_largo] - tope} tokens en silencio")
else:
    print("Sin sentence-transformers no hay modelo que truncar: el fallback TF-IDF lee "
          "el fragmento entero.")

tope del modelo: 128 tokens
fragmento más largo: [4] con 73 tokens (300 caracteres)
cabe entero


### ✏️ Ejercicio 2.1
Implementa `recuperar(pregunta, k=3)` → lista de `(score, idx_chunk)` ordenada de mayor
a menor score, usando el índice `V` y `vec_consulta(pregunta)`. La Parte 3 y el
ejercicio 3.1 la llaman **con esa firma exacta**.

In [7]:
# Ejercicio 2.1 — recuperar(pregunta, k=3) → [(score, idx_chunk), ...]
def recuperar(pregunta: str, k: int = 3):
    scores = V @ vec_consulta(pregunta)
    top = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), int(i)) for i in top]

for s, i in recuperar("¿cuántos días de vacaciones puedo pasar al año siguiente?"):
    print(f"{s:.3f} [{i}] {chunks[i][:80]}…")

0.658 [0] Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo acumulan 1…
0.532 [1] pación a través del portal interno. Los días no utilizados pueden
transferirse a…
0.444 [3] ra trabajar desde el exterior del país se requiere aprobación
de Recursos Humano…


## Parte 3 — Generación con contexto

El prompt del baseline: contexto delimitado + instrucción de responder SOLO
con base en él + permiso explícito de decir "no sé".

In [8]:
# La frase de abstención es UNA en todo el curso: la misma del Lab 02 (rag_pipeline.py)
# y del golden set del sábado. Con tres redacciones distintas en tres archivos, ningún
# detector por igualdad de cadena dispararía nunca.
ABSTENCION = "El corpus no contiene información suficiente."

PLANTILLA = """Eres un asistente de políticas internas de NimbusSoft.
Responde SOLO con base en el contexto siguiente. Si la respuesta no está en el
contexto, di exactamente: "El corpus no contiene información suficiente."
Cita el fragmento [n] que uses.

<contexto>
{contexto}
</contexto>

Pregunta: {pregunta}
Respuesta:"""

def armar_prompt(pregunta: str, k: int = 3) -> str:
    partes = [f"[{i}] {chunks[i]}" for _, i in recuperar(pregunta, k)]
    return PLANTILLA.format(contexto="\n\n".join(partes), pregunta=pregunta)

print(armar_prompt("¿puedo trabajar desde otro país?")[:900], "…")

Eres un asistente de políticas internas de NimbusSoft.
Responde SOLO con base en el contexto siguiente. Si la respuesta no está en el
contexto, di exactamente: "El corpus no contiene información suficiente."
Cita el fragmento [n] que uses.

<contexto>
[2] Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permitido hasta 3 días por semana para todos los roles
excepto soporte de infraestructura on-site. Los días remotos se coordinan con el
líder de equipo. Para trabajar desde el exterior del país se requiere aprobación
de Recursos Humano

[3] ra trabajar desde el exterior del país se requiere aprobación
de Recursos Humanos con 30 días de anticipación y un máximo de 60 días por año.

[0] Política de vacaciones de NimbusSoft.
Los empleados a tiempo completo acumulan 1.5 días de vacaciones por mes trabajado,
hasta un máximo de 18 días por año. Las vacaciones deben solicitarse c …


In [9]:
PRECAPTURADAS = {
    "transferir": 'Según [0], puedes transferir hasta un máximo de 5 días no utilizados al año siguiente.',
    "exterior": "Según el contexto, sí: requiere aprobación de RRHH con 30 días de anticipación y máximo 60 días/año.",
    "mascotas": ABSTENCION,
}

import unicodedata

def _normalizar_texto(t: str) -> str:
    """minúsculas, sin tildes, sin puntuación: así 'informacion suficiente' e
    'información suficiente.' cuentan como la misma frase."""
    t = unicodedata.normalize("NFKD", t.casefold())
    t = "".join(c for c in t if not unicodedata.combining(c))
    return " ".join("".join(c if c.isalnum() or c.isspace() else " " for c in t).split())

def se_abstuvo(respuesta: str) -> bool:
    """Detecta la abstención comparando NORMALIZADO, no por igualdad de cadena."""
    return _normalizar_texto(ABSTENCION) in _normalizar_texto(respuesta)

def _openai(prompt: str, max_salida: int = MAX_SALIDA) -> str:
    """Llamada a la API de OpenAI por HTTP, sin el SDK.

    Sin el SDK a propósito: `openai` no está en el `.venv` del curso, y un
    `import openai` dentro de un `try` que atrapa todo convierte «falta un
    paquete» en «la API falló» y manda a depurar la red. Son veinte líneas de
    `urllib` y no hay nada que instalar en treinta portátiles.
    """
    cuerpo = json.dumps({
        "model": MODELO_API,
        "messages": [{"role": "user", "content": prompt}],
        # `max_completion_tokens`, no `max_tokens`: el segundo da 400 en esta familia.
        # Y sin `temperature`: solo admite su valor por defecto, así que mandarla
        # —incluso a 0.0, que es lo que uno querría aquí— también da 400.
        "max_completion_tokens": max_salida,
    }).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions", data=cuerpo,
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
    with urllib.request.urlopen(req, timeout=90) as r:
        respuesta = json.loads(r.read())
    uso = respuesta.get("usage", {})
    detalle = uso.get("completion_tokens_details", {}) or {}
    razonados = detalle.get("reasoning_tokens")
    extra = f" ({razonados} razonando)" if razonados else ""
    print(f"   [API] {MODELO_API} · {uso.get('prompt_tokens')} tok entrada / "
          f"{uso.get('completion_tokens')} salida{extra}")
    return respuesta["choices"][0]["message"]["content"].strip()


def _ollama(prompt: str, host: str = "http://localhost:11434") -> str:
    """El mismo prompt contra un modelo local. `host` vale también para la H200."""
    cuerpo = json.dumps({"model": MODELO_LOCAL, "prompt": prompt, "stream": False}).encode()
    req = urllib.request.Request(f"{host}/api/generate", data=cuerpo,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=90) as r:
        print(f"   [local] {MODELO_LOCAL} en {host}")
        return json.loads(r.read())["response"]


def generar(prompt: str, etiqueta: str = "transferir") -> str:
    """Tres peldaños, y cada uno dice en voz alta cuál se usó.

    Que la celda imprima de dónde salió la respuesta no es cosmética: hasta el
    2026-09-23 el peldaño de la API atrapaba CUALQUIER excepción y caía al
    siguiente en silencio, así que una clave mal puesta y un modelo que no
    existe se veían igual que no tener clave — y el cuaderno seguía dando una
    respuesta, que es la peor forma de fallar.
    """
    if os.environ.get("OPENAI_API_KEY"):
        try:
            return _openai(prompt)
        except urllib.error.HTTPError as e:
            # El mensaje del proveedor dice si es la clave, el modelo o el saldo.
            detalle = e.read().decode()[:200]
            print(f"   [API] {e.code} · {detalle}")
        except urllib.error.URLError as e:
            print(f"   [API] sin red: {e.reason}")
    try:
        return _ollama(prompt)
    except Exception as e:
        print(f"   [local] no disponible ({type(e).__name__}) — "
              f"modo inspección, respuesta pre-capturada")
        return PRECAPTURADAS[etiqueta]

def responder(pregunta: str, etiqueta: str = "transferir") -> str:
    return generar(armar_prompt(pregunta), etiqueta)

print(responder("¿Cuántos días de vacaciones puedo transferir al año siguiente?", "transferir"))

   [API] gpt-5.6-terra · 241 tok entrada / 20 salida
Puedes transferir hasta un máximo de 5 días al año siguiente. [1]


### ✏️ Ejercicio 3.1 — Las tres pruebas de fuego
Ejecuta y analiza (usa las etiquetas "exterior" y "mascotas" en modo inspección):
1. Pregunta con respuesta clara: "¿puedo trabajar desde el exterior?"
2. Pregunta SIN respuesta en el corpus: "¿puedo llevar a mi mascota a la oficina?"
3. Pregunta ambigua entre documentos: "¿qué necesita aprobación del gerente?"

Para cada una anota: ¿qué chunks recuperó? ¿la respuesta se apoya en ellos?

In [10]:
# Ejercicio 3.1 — ejecución de las tres pruebas
print("1)", responder("¿Puedo trabajar desde el exterior del país?", "exterior"), "\n")
r2 = responder("¿Puedo llevar a mi mascota a la oficina?", "mascotas")
print("2)", r2, "→ ¿se abstuvo?", se_abstuvo(r2), "\n")
print("3) chunks para la ambigua:")
for s, i in recuperar("¿qué necesita aprobación del gerente?"):
    print(f"   {s:.3f} [{i}] {chunks[i][:70]}…")

   [API] gpt-5.6-terra · 250 tok entrada / 34 salida
1) Sí, con aprobación de Recursos Humanos solicitada con 30 días de anticipación. El máximo permitido es de 60 días por año. [3] 

   [API] gpt-5.6-terra · 251 tok entrada / 10 salida
2) El corpus no contiene información suficiente. → ¿se abstuvo? True 

3) chunks para la ambigua:
   0.331 [5] deben comprarse en clase económica salvo vuelos de más de
8 horas, don…
   0.270 [3] ra trabajar desde el exterior del país se requiere aprobación
de Recur…
   0.183 [2] Política de trabajo remoto de NimbusSoft.
El trabajo remoto está permi…


**Punto clave de la prueba 2:** el mejor comportamiento posible es la frase
de abstención, `ABSTENCION` («El corpus no contiene información suficiente»),
la misma que evalúa el Lab 02. Un RAG que siempre responde algo es un
generador de alucinaciones con citas decorativas.

## Cierre

Tienes un RAG completo: chunking con overlap, índice, retrieval top-k y
generación con contexto delimitado y derecho a decir "no sé".

**Mañana:** ¿este RAG es *bueno*? Métricas de retrieval (Hit Rate, MRR,
Recall@k), golden sets, y las extensiones que compiten contra tu baseline.